# Credit Card Fraud Detection under Extreme Class Imbalance
## Portfolio & Machine Learning Interview Showcase

### Executive Summary
In credit card transaction monitoring, fraudulent cases represent an **extremely tiny minority** (~$0.172\%$ in the standard Kaggle dataset). Standard accuracy is a **dangerous and misleading metric** because a dummy classifier predicting $0$ every time achieves $99.83\%$ accuracy while failing to detect a single fraud transaction.

This notebook demonstrates an end-to-end production ML workflow:
1. **Exploratory Data Analysis (EDA)** on 28 PCA features ($V_1-V_{28}$), `Time`, and `Amount`.
2. **Resampling Techniques**: Baseline Raw vs **SMOTE** (Synthetic Minority Over-sampling) vs **Cost-Sensitive Class Weighting**.
3. **Evaluations**: Precision, Recall, $F_1$, $F_2$, ROC-AUC, and **Precision-Recall AUC (PR-AUC / Average Precision)**.
4. **Financial Cost Matrix Optimization**: Tuning decision thresholds $t$ to minimize dollar loss ($C_{FN} = \$500$, $C_{FP} = \$15$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    average_precision_score, 
    precision_recall_curve,
    fbeta_score
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb

# Set plot styles
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('muted')

## 1. Synthetic Kaggle Dataset Generator
Generating realistic Kaggle-schema data ($284,807$ transaction simulation).

In [ ]:
import sys
sys.path.append('..')
from generate_data import generate_synthetic_kaggle_data

df = generate_synthetic_kaggle_data(sample_count=25000, fraud_ratio=0.00172)
print("Dataset Shape:", df.shape)
print("Target Counts:\n", df['Class'].value_counts())
print("Fraud Percentage:", f"{df['Class'].mean() * 100:.3f}%")
df.head()

## 2. Preprocessing & Stratified Splitting
Applying `RobustScaler` to `Amount` and `Time` to handle extreme numerical outliers.

In [ ]:
X = df.drop(columns=['Class', 'id'])
y = df['Class']

scaler = RobustScaler()
X[['Amount', 'Time']] = scaler.fit_transform(X[['Amount', 'Time']])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print(f"Train Fraud: {y_train.sum()} ({y_train.mean()*100:.3f}%)")
print(f"Test Fraud:  {y_test.sum()} ({y_test.mean()*100:.3f}%)")

## 3. Model Training with SMOTE Imblearn Pipeline
Building an `ImbPipeline` ensuring **SMOTE** oversampling is performed **strictly inside CV/train folds** to prevent data leakage.

In [ ]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

model_xgb = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.1, random_state=42)),
    ('clf', xgb.XGBClassifier(
        n_estimators=150, 
        max_depth=5, 
        learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr',
        random_state=42
    ))
])

model_xgb.fit(X_train, y_train)
y_pred_proba = model_xgb.predict_proba(X_test)[:, 1]

print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC (Average Precision):", average_precision_score(y_test, y_pred_proba))

## 4. Financial Cost Matrix & Decision Threshold Optimization
Evaluating total financial loss across decision thresholds $t \in [0.01, 0.99]$ where False Negative cost = $\$500$ and False Positive cost = $\$15$.

In [ ]:
sys.path.append('..')
from evaluate import optimize_threshold_cost

best_opt, df_cost = optimize_threshold_cost(y_test, y_pred_proba)
print(f"Optimal Decision Threshold: {best_opt['threshold']:.2f}")
print(f"Total Financial Loss: ${best_opt['total_cost']:,.2f}")
print(f"Dollars Saved vs Baseline: ${best_opt['savings']:,.2f} ({best_opt['savings_pct']:.1f}%)")